# 01 — Exploring and qualifying the sources

This notebook checks **concretely**, source by source, that the announced data really is
there: a text, an image, and where applicable a label. The full reasoning — why these
sources, why no scraping — is in `docs/data-source.md`.

In [1]:
from dotenv import load_dotenv

from multimodal_etl.logging_setup import setup_logging
from multimodal_etl.utils.paths import ROOT_DIR

# No `sys.path` insert: the project is installed in the environment `uv run` provides, and a
# notebook that patches the path hides which interpreter it is actually running against.
load_dotenv(ROOT_DIR / ".env")
setup_logging()

## The four sources and their access methods

A pipeline source is a module under `multimodal_etl.sources` exposing a `fetch_*`
function, plus one line in the connector table. Each goes through a different channel, and
that is deliberate: it avoids depending on a single way in.

In [2]:
from multimodal_etl.extract import _CONNECTORS

for name, connector in _CONNECTORS.items():
    print(f"{name:18s} -> {connector.__module__}")

rss                -> multimodal_etl.sources.rss
newsdata           -> multimodal_etl.sources.newsdata
fakenewsnet        -> multimodal_etl.sources.fakenewsnet
kaggle_fakeddit    -> multimodal_etl.sources.kaggle_fakeddit


## 1. RSS feeds — the multimodal backbone

RSS feeds are published by the editors *to be redistributed*: free, no key, updated
continuously. The difficulty lies elsewhere — **the image is never in the same place**
from one publisher to the next. The connector looks for it in `media:content`,
`media:thumbnail`, the attachments, then in the summary's HTML.

In [3]:
from multimodal_etl.config import ExtractionConfig
from multimodal_etl.sources.rss import fetch_rss_feed

config = ExtractionConfig()
feeds = dict(config.rss_feeds)
publications = fetch_rss_feed("bbc_news", feeds["bbc_news"], config)

with_image = [p for p in publications if p["image_url"]]
print(f"{len(publications)} publications read, {len(with_image)} of them carrying an image")

example = with_image[0]
for key in ("title", "url", "image_url", "access_method"):
    print(f"{key:14s}: {str(example[key])[:88]}")

2026-09-15 13:49:54 | INFO    | multimodal_etl.sources.rss | RSS: reading feed 'bbc_news' (https://feeds.bbci.co.uk/news/world/rss.xml)


2026-09-15 13:49:54 | INFO    | multimodal_etl.sources.rss | RSS: 29 publications collected from 'bbc_news'


29 publications read, 29 of them carrying an image


title         : Trump says AI safety fears a 'hoax' as he rejects calls for greater safeguards
url           : https://www.bbc.co.uk/news/articles/cw980n0nd0qjo?at_medium=RSS&at_campaign=rss
image_url     : https://ichef.bbci.co.uk/ace/standard/240/cpsprodpb/5749/live/ee0b1740-b0a6-11f1-9c5d-0d
access_method : rss_feed


## 2. The NewsData.io API — news already normalised

The API returns JSON with an explicit `image_url` field. In exchange it imposes a daily
quota: the connector reads a single page, and **turns itself off cleanly** when no key is
supplied rather than failing the pipeline.

In [4]:
from multimodal_etl.sources import newsdata

print("API key available:", newsdata.is_enabled())
articles = newsdata.fetch_newsdata(config)
print(f"{len(articles)} articles retrieved")
if articles:
    print("Title:", articles[0]["title"][:88])
    print("Image:", articles[0]["image_url"][:88])

API key available: False
2026-09-15 13:49:54 | INFO    | multimodal_etl.sources.newsdata | NewsData.io: no API key, source skipped.


0 articles retrieved


## 3. FakeNewsNet — a labelled dataset, but with no image

The CSVs published on GitHub hold `id, news_url, title, tweet_ids`. There is **no image**
at all: something the dataset's own description does not say outright.

Two practical consequences the connector handles:
1. the `tweet_ids` column exceeds the field size the `csv` module accepts by default — the
   limit has to be raised, or the read fails;
2. the image has to be found elsewhere: in the `og:image` tag the publisher itself puts on
   the article's page.

In [5]:
from multimodal_etl.sources import fakenewsnet

csv_path = fakenewsnet.download_csv("politifact_fake.csv", config)
rows = fakenewsnet.read_csv(csv_path)
print("Columns the file actually carries:", list(rows[0].keys()))
print(f"{len(rows)} labelled rows available")
print("A title:", rows[0]["title"][:88])

2026-09-15 13:49:54 | INFO    | multimodal_etl.sources.fakenewsnet | FakeNewsNet: 'politifact_fake.csv' already cached


Columns the file actually carries: ['id', 'news_url', 'title', 'tweet_ids']
432 labelled rows available
A title: BREAKING: First NFL Team Declares Bankruptcy Over Kneeling Thugs


### What the Open Graph enrichment actually yields

We measure what we really recover: the PolitiFact URLs date from 2016-2018 and many no
longer answer. That is a constraint to know about, not a flaw to hide — publications
without an image are dropped at the transform step.

In [6]:
from multimodal_etl.sources import opengraph

sample = [fakenewsnet._build_record(row, "politifact", "fake") for row in rows[:10]]
counters = opengraph.enrich_publications(sample, config)
counters

2026-09-15 13:50:18 | INFO    | multimodal_etl.sources.opengraph | Open Graph: 3 images recovered across 10 articles visited


{'attempted': 10, 'found': 3}

## 4. Fakeddit — the multimodal dataset from Kaggle

Fakeddit natively pairs a headline and a picture, labelled at three granularities. The file is
downloaded once by hand from Kaggle and dropped into `data/raw/kaggle/`; in its absence,
the connector reads a versioned demonstration sample of identical structure.

In [7]:
import pandas as pd

from multimodal_etl.sources import kaggle_fakeddit

kaggle_publications = kaggle_fakeddit.fetch_fakeddit(config)
kaggle_frame = pd.DataFrame(kaggle_publications)
print(f"{len(kaggle_frame)} publications loaded")
kaggle_frame["label"].value_counts()

2026-09-15 13:50:18 | INFO    | multimodal_etl.sources.kaggle_fakeddit | Fakeddit: Kaggle dataset absent, falling back to fakeddit_sample.tsv


2026-09-15 13:50:19 | INFO    | multimodal_etl.sources.kaggle_fakeddit | Fakeddit: 24 publications loaded from fakeddit_sample.tsv (demonstration sample)


24 publications loaded


label
fake    12
real    12
Name: count, dtype: int64

## Summary

The four sources complement each other: the RSS feeds and the API bring **volume and
freshness**, FakeNewsNet and Fakeddit bring **labels**. None goes through scraping: they
are all channels the data producer intended for this use.